# Replicator Dynamics -- a hands-on notebook
### PHIL 2001: Ethics and Evolutionary Games

This notebook is the **coder-friendly** version of the replicator-dynamics tool. If you
just want to explore games with sliders, use the marimo widget instead. Here you can read
and change the model itself.

All the mathematics lives in one commented file, `replicator.py`. The drawing lives in
`portraits.py`. This notebook *uses* them and explains what is going on. Cells marked
**EDIT ME** are yours to change.

## 0. Setup

If you are on Google Colab, run the cell below once to fetch the two model files.
If you are running locally in the tool's folder, they are already next to you and the
download is skipped.

In [ ]:
# Fetch the model files if they are not already here (Colab). Locally this is a no-op.
import os, urllib.request
BASE = 'https://raw.githubusercontent.com/rorysmead/phil2001-replicator/main/'  # instructor fills rorysmead/phil2001-replicator
for f in ('replicator.py', 'portraits.py'):
    if not os.path.exists(f):
        try:
            urllib.request.urlretrieve(BASE + f, f)
            print('downloaded', f)
        except Exception as e:
            print('could not fetch', f, '-- upload it manually.', e)

import numpy as np
import matplotlib.pyplot as plt
import replicator as rp
import portraits as P

## 1. The one idea

A strategy grows when it does **better than average**, and shrinks when it does worse.
In continuous time:

$$\dot{x}_i = x_i\,\big(f_i(x) - \bar f(x)\big)$$

- $x_i$ = fraction of the population playing strategy $i$ (the $x_i$ sum to 1),
- $f_i(x)$ = expected payoff to strategy $i$,
- $\bar f(x) = \sum_i x_i f_i$ = average payoff.

Here is the entire vector field, straight from `replicator.py`:

```python
def replicator_field(x, A, r=0.0):
    f = effective_payoffs(x, A, r)
    return x * (f - x @ f)
```

That is the whole model. Everything else is finding where it rests and drawing it.

## 2. A 2-strategy game: the Stag Hunt

Row = the focal player's strategy, column = the opponent's. Entry = focal's payoff.
The Stag Hunt: hunt Stag together for a big payoff, or play it safe with Hare.

In [ ]:
stag_hunt = np.array([[3.0, 0.0],
                      [1.0, 1.0]])   # S1 = Stag, S2 = Hare

P.portrait_1d(stag_hunt, labels=('Stag', 'Hare'))
plt.show()

The population lives on the line between *all Hare* and *all Stag*. Both ends are stable
(filled dots). The open dot between them is a **repeller**: the basin boundary. Start with
enough Stag hunters and you tip to all-Stag; too few and you slide to all-Hare.

The bottom panel is the *speed* $\dot x$. Where it crosses zero is a rest point; the sign
tells you the flow direction; the **slope** at the crossing is the stability (a downward
crossing attracts).

**EDIT ME:** change the Stag-Stag payoff below and re-run. How far can you lower it before
all-Stag stops being an outcome worth reaching?

In [ ]:
my_game = np.array([[2.5, 0.0],
                    [1.0, 1.0]])
P.portrait_1d(my_game, labels=('Stag', 'Hare')); plt.show()
for x, kind in rp.rest_points_classified(my_game):
    print(f'{np.round(x, 3)}  ->  {kind}')

## 3. Correlation / assortment (the spite & altruism lever)

By default players meet at random. But suppose like meets like more often than chance --
**positive assortment**, controlled by $r>0$:

$$f_i(x) = (1-r)\,(Ax)_i + r\,A_{ii}.$$

In the Prisoner's Dilemma, defection dominates under random matching. Watch what enough
assortment does to cooperation.

In [ ]:
pd = np.array([[3.0, 0.0],
               [5.0, 1.0]])   # S1 = Cooperate, S2 = Defect

for r in (0.0, 0.3, 0.6):
    fig = P.portrait_1d(pd, r=r, labels=('Cooperate', 'Defect'),
                        title=f'Prisoner\'s Dilemma, assortment r = {r}')
    plt.show()

With $r=0$ cooperation collapses. As $r$ climbs, an interior rest point appears and then
all-Cooperate becomes reachable: assortment manufactures the $r > c/b$ condition. Negative
$r$ (anti-assortment) is the mirror lever behind **spite** -- try `r=-0.3` on a game where
a strategy harms others.

**EDIT ME:** find the smallest $r$ at which the PD gains an interior rest point.

## 4. Three strategies: Rock-Paper-Scissors on the simplex

With three strategies the population lives on a triangle (the 2-simplex). Standard,
zero-sum RPS has a famous feature: the interior point is a **center** -- orbits circle it
forever without converging.

In [ ]:
rps = np.array([[ 0.0, -1.0,  1.0],
                [ 1.0,  0.0, -1.0],
                [-1.0,  1.0,  0.0]])
P.portrait_simplex(rps, labels=('Rock', 'Paper', 'Scissors')); plt.show()
print('interior point is a:', rp.classify([1/3, 1/3, 1/3], rps))

Tip the payoffs slightly and the center becomes a genuine attractor (a spiral sink) --
the `RPS (attracting)` preset. This is the kind of qualitative change that a phase
portrait makes visible at a glance.

In [ ]:
rps_att = rp.GAMES_3x3['RPS (attracting)']
P.portrait_simplex(rps_att, labels=('Rock', 'Paper', 'Scissors')); plt.show()
print('interior point is now:', rp.classify([1/3, 1/3, 1/3], rps_att))

## 5. Basins of attraction, measured by simulation

"If we start from a random population, where do we usually end up?" We drop many random
starting points, run each to its resting place, and count. The answer is statistical, so
we report a 95% confidence interval.

In [ ]:
est = rp.estimate_basins(stag_hunt, n_samples=3000, seed=0)
for a in est['attractors']:
    lo, hi = 100*a['ci_low'], 100*a['ci_high']
    print(f"{np.round(a['state'],2)}  ({a['type']}):  {100*a['fraction']:.1f}%  "
          f"[95% CI {lo:.1f}-{hi:.1f}%]")
if est['unresolved']:
    print(f"unresolved: {100*est['unresolved']:.1f}%")

**EDIT ME:** increase `n_samples`. Watch the confidence interval shrink -- that is the
$1/\sqrt{N}$ law of large numbers doing its work.

> **Why Monte Carlo, and not an exact formula?** A basin boundary is the *separatrix* --
> the stable manifold of the saddle between attractors. For a general (nonlinear) game
> that curve has no closed form, so exact basins in a formula are impossible in general
> (a fact about nonlinear ODEs, not a gap in the theory). Monte Carlo is the principled
> estimate; the confidence interval is honest about the sampling.
>
> **Degenerate games.** Basins belong only to *asymptotically stable* objects. A neutral
> center (standard RPS) has none -- orbits circle forever. And if a whole *line/region* of
> the simplex is rest points (e.g. two interchangeable strategies), the tool now finds it
> as an attracting/repelling **set** (`rp.rest_objects(dyn, A)`), classified by its
> transverse stability, and basins count it as a single attractor.

## 6. Two populations (asymmetric games)

Some games have two *roles* with different payoffs -- buyer/seller, or the two sides of the
Battle of the Sexes. The state is now a point $(x, y)$ in the unit square: $x$ = population
1's share of its first strategy, $y$ = population 2's.

In [ ]:
A, B = rp.GAMES_BIMATRIX['Battle of the Sexes']
P.portrait_square(A, B, labels1=('Opera', 'Fight'), labels2=('Opera', 'Fight'))
plt.show()

Flow runs to the two coordinated corners; the mixed equilibrium in the middle is a saddle.
Try `'Matching Pennies'` instead -- no pure equilibrium, so the flow *circles* the center,
the two-population echo of Rock-Paper-Scissors.

The **same five dynamics apply here** too: pass `dyn=rp.DYNAMICS[key]` to `portrait_square`.
Under replicator the Matching Pennies center is neutral (closed orbits); under
replicator-mutator mutation damps it into a stable spiral -- exactly as in the
single-population RPS. `rp.bimatrix_rest_points(dyn, A, B)` lists the square's fixed points
with their stability.

In [ ]:
# Matching Pennies: neutral center under replicator, stabilised by mutation.
Amp, Bmp = rp.GAMES_BIMATRIX['Matching Pennies']
for key, kw in [('replicator', {}), ('replmut', {'mu': 0.05})]:
    pts = rp.bimatrix_rest_points(rp.DYNAMICS[key], Amp, Bmp, **kw)
    interior = [(z, t) for z, t in pts if 0.1 < z[0] < 0.9 and 0.1 < z[1] < 0.9]
    print(f"{rp.DYNAMICS[key].name:22s}: interior {interior[0][0].round(2)} is {interior[0][1]}")

## 7. Alternative dynamics

The replicator dynamic is one story about how frequencies change; it is not the only one.
A *dynamic* encodes a behavioural assumption, and different assumptions give different
pictures for the SAME game. A conclusion that survives a change of dynamic is robust; one
that does not is an artefact of the dynamic. The tool ships five, grouped by family:

| key | dynamic | family | can revive extinct strategies? |
|---|---|---|---|
| `replicator` | continuous replicator | imitative | no |
| `discrete` | discrete-time replicator | imitative | no |
| `bnn` | Brown-von Neumann-Nash | innovative | **yes** |
| `logit` | logit best-response | perturbed best response | (interior) |
| `replmut` | replicator-mutator | selection + mutation | (interior) |

Assortment `r` composes with every one of them (it changes the payoff `f_i`). Pass a
dynamic with `dyn=rp.DYNAMICS[key]`, and any parameter (`beta` for logit, `mu` for
replicator-mutator) as a keyword.

In [ ]:
# The same RPS game under four dynamics -- watch the interior point change character.
rps = rp.GAMES_3x3['Rock-Paper-Scissors']
for key, kw in [('replicator', {}), ('discrete', {}), ('bnn', {}), ('replmut', {'mu': 0.05})]:
    dyn = rp.DYNAMICS[key]
    print(f"{dyn.name:28s}: interior point is {rp.classify_dyn(dyn, rps, [1/3,1/3,1/3], **kw)}")
    P.portrait_simplex(rps, dyn=dyn, labels=('R','P','S'), **kw); plt.show()

Continuous replicator: a neutral **center** (closed orbits). Discrete time: the center
**repels** and orbits spiral out to the boundary -- the classic discrete-replicator
artefact, and exactly why a phase-portrait tool should default to continuous time. BNN:
the center **attracts** (it is the Nash equilibrium). Replicator-mutator: mutation damps
the orbits into a slow stable spiral.

In [ ]:
# BNN can REVIVE an extinct strategy; the imitative replicator cannot.
start = [1.0, 0.0, 0.0]   # pure Rock -- Paper and Scissors are extinct
print('replicator from pure Rock ->', np.round(rp.endpoint_dyn(rp.DYNAMICS['replicator'], start, rps), 3))
print('BNN        from pure Rock ->', np.round(rp.endpoint_dyn(rp.DYNAMICS['bnn'], start, rps), 3))

In [ ]:
# Logit: rest points are quantal-response equilibria that sharpen to Nash as beta grows.
pd = rp.GAMES_2x2["Prisoner's Dilemma"]
for beta in [0.5, 2.0, 10.0]:
    pts = rp.rest_points_dyn(rp.DYNAMICS['logit'], pd, beta=beta)
    print(f'beta={beta:4.1f}: rest point = {np.round(pts[0], 3).tolist()}  (-> all-Defect as beta grows)')

## 8. Exercises

1. **Hawk-Dove.** Load `rp.GAMES_2x2['Hawk-Dove']`. Where is the interior ESS, and what
   assortment $r$ pushes the population toward *all* Dove?
2. **Build a game.** Write a 3x3 payoff matrix with two stable pure strategies and estimate
   their basins. Which corner wins more often, and why?
3. **Spite.** Design a 2-strategy game with a harmful strategy and use $r<0$ to give it a
   foothold. What is the smallest population share it needs (its invasion barrier)?
4. **Break something.** Add the same constant to every payoff in any game. Confirm the
   portrait does not move -- then read the note in `replicator.py` on why the discrete-time
   version would have moved.